# 11.4 Time Series Visualization and Analysis

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 11: Time Series Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-11-time-series-machine-learning/11.04_time-series-visualization-and-analysis.ipynb)

Companion code for Section 11.4 of the book. Run the cells in order from top to bottom.

**Example 11.4.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import acf, pacf
import warnings; warnings.filterwarnings('ignore')

# ── Generate different ACF patterns for illustration ──────────────
np.random.seed(42)
n = 300

# AR(2) process: yₜ = 0.7yₜ₋₁ - 0.3yₜ₋₂ + εₜ
y_ar2 = np.zeros(n)
for t in range(2, n):
    y_ar2[t] = 0.7*y_ar2[t-1] - 0.3*y_ar2[t-2] + np.random.normal(0, 1)

# MA(2) process: yₜ = εₜ + 0.6εₜ₋₁ - 0.4εₜ₋₂
eps = np.random.normal(0, 1, n+2)
y_ma2 = np.array([eps[t] + 0.6*eps[t-1] - 0.4*eps[t-2] for t in range(2, n+2)])

# Non-stationary: random walk yₜ = yₜ₋₁ + εₜ
y_rw  = np.cumsum(np.random.normal(0, 1, n))

# Seasonal (period=7) + trend
t_arr   = np.arange(n)
y_seas  = 0.5*t_arr + 10*np.sin(2*np.pi*t_arr/7) + np.random.normal(0,2,n)

fig, axes = plt.subplots(4, 3, figsize=(18, 16))
fig.suptitle('ACF and PACF Patterns - Model Identification Guide',
            fontsize=14, fontweight='bold')

processes = [
    ('AR(2): yt=0.7yt₋₁-0.3yt₋₂+ε', y_ar2, 'ACF: tails off\nPACF: cuts at lag 2 → AR(2)'),
    ('MA(2): yt=ε+0.6ε₋₁-0.4ε₋₂',   y_ma2, 'ACF: cuts at lag 2 → MA(2)\nPACF: tails off'),
    ('Random Walk (non-stationary)',   y_rw,  'ACF: slow decay → needs differencing'),
    ('Seasonal (period=7) + Trend',   y_seas,'ACF: spikes at lags 7,14,21 → seasonal'),
]

for row, (name, y, interpretation) in enumerate(processes):
    # Time plot
    ax = axes[row, 0]
    ax.plot(y[:100], color='#1976D2', lw=1.5)
    ax.set_title(f'{name}', fontsize=9, fontweight='bold')
    ax.set_xlabel('Time'); ax.set_ylabel('Value'); ax.grid(alpha=0.3)

    # ACF
    ax = axes[row, 1]
    plot_acf(y, lags=30, ax=ax, color='#1976D2', alpha=0.05, zero=False)
    ax.set_title(f'ACF - {interpretation.split(chr(10))[0]}', fontsize=9)
    ax.set_xlabel('Lag'); ax.grid(alpha=0.3)

    # PACF
    ax = axes[row, 2]
    plot_pacf(y, lags=30, ax=ax, method='ywm', color='#E53935', alpha=0.05, zero=False)
    ax.set_title(f'PACF - {interpretation.split(chr(10))[1] if chr(10) in interpretation else ""}'
               , fontsize=9)
    ax.set_xlabel('Lag'); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('acf_pacf_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Seasonality Analysis: multiple seasonal patterns ──────────────
print('=== Autocorrelation at Key Lags ===')
acf_values = acf(y_seas, nlags=50)
seasonal_lags = [7, 14, 21, 28, 35]
print('Seasonal lags analysis (period=7):')
for lag in seasonal_lags:
    print(f'  ACF(lag={lag:>2}): {acf_values[lag]:+.4f}')